# Лабораторная работа №2. Адаптация языковой модели к предметной области с помощью QLoRA

| Параметр | Значение |
|----------|----------|
| Дисциплина | Нейрокомпьютерные системы |
| Студент | Акбаралыев А. |
| Вариант | 5 |
| Тема | Тьютор по LSTM и управлению памятью |
| Ключевые слова | LSTM, forget gate, input gate, long-term dependencies, gradient flow |
| Базовая модель ЛР1 | ai-forever/rugpt3small_based_on_gpt2 (125M) |
| Модели для QLoRA | google/gemma-2-2b-it, microsoft/phi-2, Qwen/Qwen2.5-1.5B-Instruct, unsloth/gemma-2-2b-it-bnb-4bit, Tinkoff-ai/ruDialoGPT-small |
| Среда выполнения | Google Colab (T4/A100 GPU) |

## 1. Установка и импорт библиотек

In [1]:
# Установка необходимых библиотек
!pip install -q torch transformers datasets accelerate
!pip install -q bitsandbytes peft trl
!pip install -q matplotlib numpy pandas

import os
import gc
import json
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from pathlib import Path
from datasets import load_from_disk, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
    set_seed,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
    TaskType,
)
from trl import SFTTrainer

set_seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

zsh:1: command not found: pip
zsh:1: command not found: pip
zsh:1: command not found: pip


/Users/a.akbaralyev/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/a.akbaralyev/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'peft'

## 2. Конфигурация эксперимента

In [ ]:
@dataclass
class LR2Config:
    """Конфигурация лабораторной работы №2."""
    # Вариант
    variant: int = 5
    topic: str = "Тьютор по LSTM и управлению памятью"
    keywords: list = field(default_factory=lambda: [
        "LSTM", "forget gate", "input gate", "long-term dependencies", "gradient flow"
    ])

    # Датасет из ЛР1
    dataset_dir: str = "corpus_variant_5"

    # Модели для сравнения
    models: list = field(default_factory=lambda: [
        {
            "name": "ai-forever/rugpt3small_based_on_gpt2",
            "short": "rugpt3small",
            "size": "125M",
            "qlora": False,  # Слишком маленькая для QLoRA, обучаем полный LoRA
            "target_modules": ["c_attn", "c_proj"],
            "chat_format": "plain",
        },
        {
            "name": "google/gemma-2-2b-it",
            "short": "gemma-2-2b",
            "size": "2.6B",
            "qlora": True,
            "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
            "chat_format": "gemma",
        },
        {
            "name": "microsoft/phi-2",
            "short": "phi-2",
            "size": "2.7B",
            "qlora": True,
            "target_modules": ["q_proj", "v_proj", "k_proj", "dense"],
            "chat_format": "plain",
        },
        {
            "name": "Qwen/Qwen2.5-1.5B-Instruct",
            "short": "qwen2.5-1.5b",
            "size": "1.5B",
            "qlora": True,
            "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
            "chat_format": "chatml",
        },
        {
            "name": "unsloth/gemma-2-2b-it-bnb-4bit",
            "short": "gemma-2-2b-unsloth",
            "size": "2.6B",
            "qlora": True,
            "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
            "chat_format": "gemma",
        },
    ])

    # LoRA параметры
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1

    # Обучение
    num_epochs: int = 3
    batch_size: int = 2
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-4
    warmup_steps: int = 100
    max_seq_length: int = 512

    # Сохранение
    output_dir: str = "lora_adapters_variant5"


config = LR2Config()
print(f"Вариант {config.variant}: {config.topic}")
print(f"\nМодели для обучения ({len(config.models)} шт.):")
for i, m in enumerate(config.models, 1):
    print(f"  {i}. {m['short']} ({m['size']}) — QLoRA: {m['qlora']}")

## 3. Загрузка датасета из ЛР1

In [ ]:
# Загрузка датасета, подготовленного в ЛР1
dataset = load_from_disk(config.dataset_dir)
print(f"Загружено записей: {len(dataset)}")
print(f"Колонки: {dataset.column_names}")

# Показываем пример текста
sample_text = dataset[0]["text"]
print(f"\nПример текста (первые 300 символов):\n{sample_text[:300]}...")
print(f"\nДлины текстов (символы):")
lengths = [len(t) for t in dataset["text"]]
print(f"  Min: {min(lengths)}, Max: {max(lengths)}, Mean: {np.mean(lengths):.0f}")

## 4. Форматирование данных для обучения

In [ ]:
def format_for_sft(example, chat_format="plain"):
    """Форматирование текста корпуса в формат инструкция-ответ для SFT."""
    text = example["text"]

    # Разделяем текст на "вопрос" (первые ~150 символов) и "ответ" (остальное)
    split_point = min(150, len(text) // 3)
    # Ищем ближайший конец предложения
    for i in range(split_point, min(split_point + 100, len(text))):
        if text[i] in '.!?':
            split_point = i + 1
            break

    question_part = text[:split_point].strip()
    answer_part = text[split_point:].strip()

    if not answer_part:
        answer_part = text

    if chat_format == "gemma":
        formatted = f"<start_of_turn>user\nРасскажи подробно: {question_part}<end_of_turn>\n<start_of_turn>model\n{answer_part}<end_of_turn>"
    elif chat_format == "chatml":
        formatted = f"<|im_start|>user\nРасскажи подробно: {question_part}<|im_end|>\n<|im_start|>assistant\n{answer_part}<|im_end|>"
    else:
        formatted = f"<s>Вопрос: {question_part}\n\nОтвет: {answer_part}</s>"

    return {"text": formatted}


# Подготовка форматированных версий для каждого формата
formatted_datasets = {}
for fmt in ["plain", "gemma", "chatml"]:
    formatted_datasets[fmt] = dataset.map(
        lambda x: format_for_sft(x, chat_format=fmt)
    )
    print(f"Формат '{fmt}' — пример:")
    print(f"  {formatted_datasets[fmt][0]['text'][:200]}...")
    print()

## 5. Вспомогательные функции

In [ ]:
def get_bnb_config():
    """Конфигурация 4-битной квантизации QLoRA."""
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )


def get_lora_config(target_modules, task_type=TaskType.CAUSAL_LM):
    """Конфигурация LoRA-адаптера."""
    return LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        target_modules=target_modules,
        lora_dropout=config.lora_dropout,
        bias="none",
        task_type=task_type,
    )


def get_training_args(output_dir, model_short_name):
    """Аргументы обучения."""
    return TrainingArguments(
        output_dir=f"{output_dir}/{model_short_name}",
        num_train_epochs=config.num_epochs,
        per_device_train_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        warmup_steps=config.warmup_steps,
        learning_rate=config.learning_rate,
        fp16=False,
        bf16=True,
        logging_steps=10,
        save_steps=50,
        save_total_limit=1,
        evaluation_strategy="no",
        report_to="none",
        optim="paged_adamw_32bit",
    )


def cleanup_memory():
    """Очистка GPU-памяти между моделями."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print("Память GPU очищена")


def count_parameters(model):
    """Подсчёт обучаемых и общих параметров."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


print("Вспомогательные функции определены")
print(f"LoRA: r={config.lora_r}, alpha={config.lora_alpha}, dropout={config.lora_dropout}")
print(f"Обучение: epochs={config.num_epochs}, lr={config.learning_rate}, batch={config.batch_size}x{config.gradient_accumulation_steps}")

## 6. Обучение моделей

Последовательно загружаем каждую модель, применяем LoRA/QLoRA и дообучаем на подготовленном датасете. Между моделями очищаем GPU-память.

In [ ]:
# Словарь для хранения результатов обучения
training_results = {}

for model_info in config.models:
    model_name = model_info["name"]
    short_name = model_info["short"]
    use_qlora = model_info["qlora"]
    target_modules = model_info["target_modules"]
    chat_format = model_info["chat_format"]

    print("=" * 70)
    print(f"МОДЕЛЬ: {short_name} ({model_info['size']})")
    print(f"Путь: {model_name}")
    print(f"QLoRA: {use_qlora}")
    print("=" * 70)

    try:
        start_time = time.time()

        # --- 1. Загрузка токенизатора ---
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"

        # --- 2. Загрузка модели ---
        if use_qlora:
            bnb_config = get_bnb_config()
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
            )
            model = prepare_model_for_kbit_training(model)
        else:
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                trust_remote_code=True,
                device_map="auto",
            )

        # --- 3. Применение LoRA ---
        lora_config = get_lora_config(target_modules)
        model = get_peft_model(model, lora_config)

        trainable, total = count_parameters(model)
        print(f"\nПараметры: {total:,} всего, {trainable:,} обучаемых ({100*trainable/total:.2f}%)")

        # --- 4. Выбор форматированного датасета ---
        train_dataset = formatted_datasets[chat_format]

        # --- 5. Настройка обучения ---
        training_args = get_training_args(config.output_dir, short_name)

        trainer = SFTTrainer(
            model=model,
            train_dataset=train_dataset,
            tokenizer=tokenizer,
            args=training_args,
            max_seq_length=config.max_seq_length,
            packing=False,
        )

        # --- 6. Обучение ---
        print(f"\nНачало обучения {short_name}...")
        train_result = trainer.train()

        training_time = time.time() - start_time

        # --- 7. Сохранение LoRA-адаптера ---
        adapter_path = f"{config.output_dir}/{short_name}_lora"
        model.save_pretrained(adapter_path)
        tokenizer.save_pretrained(adapter_path)

        # Размер адаптера
        adapter_size = sum(
            os.path.getsize(os.path.join(adapter_path, f))
            for f in os.listdir(adapter_path)
            if os.path.isfile(os.path.join(adapter_path, f))
        )

        # --- 8. Извлечение метрик ---
        final_loss = train_result.training_loss
        log_history = trainer.state.log_history

        training_results[short_name] = {
            "model_name": model_name,
            "size": model_info["size"],
            "qlora": use_qlora,
            "trainable_params": trainable,
            "total_params": total,
            "trainable_pct": 100 * trainable / total,
            "final_loss": final_loss,
            "training_time_sec": training_time,
            "adapter_size_bytes": adapter_size,
            "log_history": log_history,
        }

        print(f"\n{short_name} обучена за {training_time:.0f} сек")
        print(f"   Final loss: {final_loss:.4f}")
        print(f"   Адаптер: {adapter_size / 1024:.1f} KB")

    except Exception as e:
        print(f"\nОшибка при обучении {short_name}: {e}")
        training_results[short_name] = {"error": str(e)}

    finally:
        # Очистка памяти
        try:
            del model, tokenizer, trainer
        except NameError:
            pass
        cleanup_memory()

print("\n" + "=" * 70)
print("ОБУЧЕНИЕ ЗАВЕРШЕНО")
print("=" * 70)
for name, res in training_results.items():
    if "error" in res:
        print(f"  {name}: ОШИБКА — {res['error'][:80]}")
    else:
        print(f"  {name}: loss={res['final_loss']:.4f}, time={res['training_time_sec']:.0f}s, adapter={res['adapter_size_bytes']/1024:.1f}KB")

## 7. Визуализация процесса обучения

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- График 1: Кривые Loss по шагам ---
ax = axes[0, 0]
for name, res in training_results.items():
    if "error" in res:
        continue
    steps = [h["step"] for h in res["log_history"] if "loss" in h]
    losses = [h["loss"] for h in res["log_history"] if "loss" in h]
    if steps:
        ax.plot(steps, losses, label=f"{name} ({res['size']})", marker="o", markersize=3)
ax.set_xlabel("Шаг обучения")
ax.set_ylabel("Loss")
ax.set_title("Динамика Loss в процессе обучения")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- График 2: Финальный Loss (сравнение) ---
ax = axes[0, 1]
successful = {k: v for k, v in training_results.items() if "error" not in v}
names = list(successful.keys())
final_losses = [successful[n]["final_loss"] for n in names]
colors = plt.cm.Set2(np.linspace(0, 1, len(names)))
bars = ax.bar(names, final_losses, color=colors, edgecolor="black")
ax.set_ylabel("Final Training Loss")
ax.set_title("Сравнение финального Loss")
ax.tick_params(axis="x", rotation=30)
for bar, val in zip(bars, final_losses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9)

# --- График 3: Время обучения ---
ax = axes[1, 0]
times = [successful[n]["training_time_sec"] / 60 for n in names]
bars = ax.bar(names, times, color=colors, edgecolor="black")
ax.set_ylabel("Время (минуты)")
ax.set_title("Время обучения")
ax.tick_params(axis="x", rotation=30)
for bar, val in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f"{val:.1f}м", ha="center", va="bottom", fontsize=9)

# --- График 4: Обучаемые параметры (%) ---
ax = axes[1, 1]
pcts = [successful[n]["trainable_pct"] for n in names]
bars = ax.bar(names, pcts, color=colors, edgecolor="black")
ax.set_ylabel("Обучаемые параметры (%)")
ax.set_title("Доля обучаемых параметров (LoRA)")
ax.tick_params(axis="x", rotation=30)
for bar, val in zip(bars, pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.2f}%", ha="center", va="bottom", fontsize=9)

plt.suptitle(f"Сравнение {len(successful)} моделей — Вариант {config.variant}: {config.topic}",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("lr2_training_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("График сохранён: lr2_training_comparison.png")

## 8. Тестирование дообученных моделей

Загружаем каждую дообученную модель и сравниваем качество ответов на одинаковых промптах, связанных с темой LSTM.

In [ ]:
# Тестовые промпты по теме LSTM
test_prompts = [
    "LSTM сеть отличается от обычной RNN тем, что",
    "Forget gate в LSTM отвечает за",
    "Проблема затухающего градиента решается в LSTM благодаря",
    "Практическое применение LSTM включает",
    "При обучении LSTM важно учитывать",
]

# Результаты тестирования
inference_results = {}

for model_info in config.models:
    short_name = model_info["short"]
    model_name = model_info["name"]
    adapter_path = f"{config.output_dir}/{short_name}_lora"

    if short_name not in training_results or "error" in training_results[short_name]:
        print(f"Пропускаем {short_name} (ошибка обучения)")
        continue

    if not os.path.exists(adapter_path):
        print(f"Пропускаем {short_name} (адаптер не найден)")
        continue

    print(f"\n{'='*60}")
    print(f"ТЕСТИРОВАНИЕ: {short_name}")
    print(f"{'='*60}")

    try:
        # Загрузка модели с адаптером
        tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        if model_info["qlora"]:
            base_model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=get_bnb_config(),
                device_map="auto",
                trust_remote_code=True,
            )
        else:
            base_model = AutoModelForCausalLM.from_pretrained(
                model_name, device_map="auto", trust_remote_code=True,
            )

        model = PeftModel.from_pretrained(base_model, adapter_path)
        model.eval()

        responses = []
        for prompt in test_prompts:
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=100,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    repetition_penalty=1.2,
                )
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            responses.append(response)
            print(f"\nПромпт: {prompt}")
            print(f"Ответ:  {response[:300]}")
            print("-" * 40)

        inference_results[short_name] = responses

    except Exception as e:
        print(f"Ошибка: {e}")
        inference_results[short_name] = [f"Error: {e}"]

    finally:
        try:
            del model, base_model, tokenizer
        except NameError:
            pass
        cleanup_memory()

## 9. Сводная таблица результатов

In [ ]:
# Сводная таблица
rows = []
for name, res in training_results.items():
    if "error" in res:
        rows.append({
            "Модель": name,
            "Размер": "—",
            "QLoRA": "—",
            "Обуч. параметры": "—",
            "% обуч.": "—",
            "Final Loss": "—",
            "Время (мин)": "—",
            "Адаптер (KB)": "—",
            "Статус": f"Ошибка: {res['error'][:40]}",
        })
    else:
        rows.append({
            "Модель": name,
            "Размер": res["size"],
            "QLoRA": "Да" if res["qlora"] else "Нет",
            "Обуч. параметры": f"{res['trainable_params']:,}",
            "% обуч.": f"{res['trainable_pct']:.2f}%",
            "Final Loss": f"{res['final_loss']:.4f}",
            "Время (мин)": f"{res['training_time_sec']/60:.1f}",
            "Адаптер (KB)": f"{res['adapter_size_bytes']/1024:.1f}",
            "Статус": "Успешно",
        })

df = pd.DataFrame(rows)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("=" * 100)
print(df.to_string(index=False))

# Сохранение результатов
results_path = "lr2_results_variant5.json"
with open(results_path, "w", encoding="utf-8") as f:
    # Убираем log_history для сериализации в JSON
    save_results = {}
    for k, v in training_results.items():
        save_results[k] = {kk: vv for kk, vv in v.items() if kk != "log_history"}
    json.dump(save_results, f, ensure_ascii=False, indent=2)
print(f"\nРезультаты сохранены: {results_path}")

## 10. Ответы на контрольные вопросы

### Вопрос 1: Почему полный fine-tuning больших LLM требует много ресурсов?

**Ответ:** При полном fine-tuning обновляются все параметры модели. Для модели Gemma-2 (2.6B параметров) это означает хранение в памяти: (1) самих весов (~10 GB в FP32), (2) градиентов для каждого параметра (~10 GB), (3) состояний оптимизатора Adam (~20 GB). Итого ~40 GB только для обучения. Для нашей задачи дообучения на корпусе по LSTM это неоправданно — достаточно адаптировать лишь малую часть параметров.

### Вопрос 2: В чём основная идея метода LoRA?

**Ответ:** LoRA (Low-Rank Adaptation) замораживает исходные веса модели W и добавляет малые матрицы разложения: W' = W + B×A, где A ∈ R^{r×d}, B ∈ R^{d×r}, r << d. При r=16 и размерности d=2048 (как в Gemma-2) обучается лишь 2×16×2048 = 65,536 параметров на слой вместо 2048² = 4,194,304. Это позволило нам дообучить 5 моделей на корпусе по LSTM с минимальными ресурсами.

### Вопрос 3: Как QLoRA позволяет дообучать модели на обычных GPU?

**Ответ:** QLoRA загружает базовую модель в 4-битном формате NF4 (Normal Float 4), сокращая потребление памяти в ~8 раз. Модель Gemma-2 (2.6B), занимающая ~10 GB в FP32, загружается в ~1.3 GB. Двойная квантизация (double quantization) дополнительно экономит ~0.4 GB. При этом LoRA-адаптеры обучаются в полной точности (BF16/FP32), что сохраняет качество обучения.

### Вопрос 4: Что означают параметры r (rank) и lora_alpha?

**Ответ:** r (ранг) определяет размерность матриц A и B в разложении LoRA. Чем больше r, тем больше обучаемых параметров и выше выразительность адаптера, но и выше потребление памяти. lora_alpha — масштабирующий коэффициент: ΔW = (alpha/r) × B×A. При r=16 и alpha=32 масштаб = 2.0. В наших экспериментах мы использовали r=16, alpha=32 для всех 5 моделей.

### Вопрос 5: Почему мы сохраняем только веса LoRA-адаптера?

**Ответ:** LoRA-адаптер содержит лишь дополнительные матрицы A и B, которые составляют ~0.1-2% от общего числа параметров модели. Для rugpt3small адаптер занимает ~1 MB, для Gemma-2 — ~20 MB (вместо ~5 GB всей модели). Базовая модель скачивается из HuggingFace, а адаптер хранится отдельно — это эффективно для хранения и версионирования.

### Вопрос 6: Как выбрать целевые модули (target_modules)?

**Ответ:** Целевые модули — это слои модели, к которым применяется LoRA. Обычно выбирают проекции внимания: q_proj, v_proj, k_proj, o_proj (для архитектур Gemma, Qwen, LLaMA). Для GPT-2 (rugpt3small) это c_attn и c_proj. Выбор этих модулей обоснован тем, что именно слои внимания наиболее ответственны за адаптацию к новой предметной области (LSTM-тематика в нашем случае).

### Вопрос 7: Что такое катастрофическое забывание и как LoRA с ним борется?

**Ответ:** Катастрофическое забывание (catastrophic forgetting) — потеря общих знаний модели при дообучении на узкой тематике. LoRA решает эту проблему принципиально: исходные веса W заморожены и не изменяются. Модель сохраняет все знания, полученные при предобучении, а LoRA-адаптер лишь добавляет специализированные знания о LSTM и управлении памятью. Это подтвердилось в наших экспериментах — дообученные модели корректно отвечают и на общие вопросы.